# 🧠 Notebook 2: Concept & Term Extraction

Extract domain concepts from PDF text using:
- **Named Entity Recognition (NER)** via spaCy
- **Noun Phrase Chunking**
- **TF-IDF Key Term Ranking**

---

In [ ]:
!pip install -q spacy scikit-learn PyMuPDF
!python -m spacy download en_core_web_sm -q
!git clone https://github.com/YOUR_USERNAME/ontology-builder.git 2>/dev/null || true
import sys; sys.path.insert(0, 'ontology-builder')

## 1. Load Extracted Text

Load from Notebook 1 output or extract fresh.

In [ ]:
import json, os

# Load from previous notebook
if os.path.exists('output/extracted_texts.json'):
    with open('output/extracted_texts.json') as f:
        extracted_data = json.load(f)
    texts = [d['full_text'] for d in extracted_data]
    print(f'Loaded {len(texts)} documents')
else:
    # Demo text for standalone usage
    texts = [
        """Machine learning is a subset of artificial intelligence that enables 
        systems to learn from data. Deep learning, a branch of machine learning, 
        uses neural networks with many layers. Convolutional Neural Networks (CNNs) 
        are used for image recognition tasks. Recurrent Neural Networks (RNNs) 
        process sequential data like text and speech. Natural Language Processing 
        is an application of deep learning. Companies such as Google, Microsoft, 
        and OpenAI are leaders in AI research. Transfer learning allows models 
        pre-trained on large datasets to be fine-tuned for specific tasks. 
        Reinforcement learning trains agents through reward signals.""",
        
        """Supervised learning algorithms include decision trees, support vector 
        machines, and random forests. Unsupervised learning methods such as 
        k-means clustering and principal component analysis find patterns in 
        unlabeled data. Semi-supervised learning combines labeled and unlabeled 
        data. Gradient descent is the primary optimization algorithm used in 
        training neural networks. Backpropagation computes gradients efficiently. 
        Transformers are a type of neural network architecture that uses 
        self-attention mechanisms."""
    ]
    print(f'Using {len(texts)} demo documents')

## 2. Text Preprocessing

In [ ]:
from src.preprocessor import TextPreprocessor

preprocessor = TextPreprocessor(min_segment_length=20)

all_segments = []
for i, text in enumerate(texts):
    segments = preprocessor.preprocess(text, source_doc=f'doc_{i}')
    all_segments.extend(segments)

print(f'Total segments: {len(all_segments)}')
for seg in all_segments[:3]:
    print(f'  [{seg.segment_type}] {seg.text[:80]}...')

## 3. Named Entity Recognition

In [ ]:
import spacy
from collections import Counter

nlp = spacy.load('en_core_web_sm')

full_text = ' '.join(seg.text for seg in all_segments)
doc = nlp(full_text)

# Count entities by type
entity_counter = Counter()
for ent in doc.ents:
    entity_counter[(ent.text, ent.label_)] += 1

print('Top Named Entities:')
print(f'{"Entity":<30} {"Type":<10} {"Count"}')
print('-' * 50)
for (text, label), count in entity_counter.most_common(20):
    print(f'{text:<30} {label:<10} {count}')

## 4. Noun Phrase Extraction

In [ ]:
# Extract and rank noun phrases
np_counter = Counter()
for chunk in doc.noun_chunks:
    text = chunk.text.strip()
    if len(text) > 3 and chunk.root.pos_ in ('NOUN', 'PROPN'):
        np_counter[text.lower()] += 1

print('Top Noun Phrases:')
print(f'{"Phrase":<40} {"Count"}')
print('-' * 50)
for phrase, count in np_counter.most_common(25):
    print(f'{phrase:<40} {count}')

## 5. TF-IDF Key Terms

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

segment_texts = [seg.text for seg in all_segments if len(seg.text) > 30]

if len(segment_texts) >= 2:
    vectorizer = TfidfVectorizer(
        max_features=50,
        ngram_range=(1, 3),
        stop_words='english',
        min_df=1
    )
    tfidf_matrix = vectorizer.fit_transform(segment_texts)
    features = vectorizer.get_feature_names_out()
    scores = tfidf_matrix.sum(axis=0).A1
    
    ranked = sorted(zip(features, scores), key=lambda x: x[1], reverse=True)
    
    print('Top TF-IDF Terms:')
    print(f'{"Term":<35} {"Score"}')
    print('-' * 45)
    for term, score in ranked[:20]:
        print(f'{term:<35} {score:.4f}')
else:
    print('Need at least 2 segments for TF-IDF. Try with more documents.')

## 6. Full Concept Extraction Pipeline

In [ ]:
from src.concept_extractor import ConceptExtractor

extractor = ConceptExtractor(
    spacy_model='en_core_web_sm',
    min_frequency=1,
    max_concepts=100,
    tfidf_top_n=50
)

concepts = extractor.extract(all_segments)

print(f'\nExtracted {len(concepts)} concepts:\n')
print(f'{"Class Name":<30} {"Label":<30} {"Freq"}')
print('=' * 65)
for c in concepts[:30]:
    print(f'{c.name:<30} {c.label:<30} {c.frequency}')

## 7. Save Concepts for Next Notebook

In [ ]:
os.makedirs('output', exist_ok=True)

concepts_data = [
    {'name': c.name, 'label': c.label, 'frequency': c.frequency, 'synonyms': c.synonyms}
    for c in concepts
]

with open('output/concepts.json', 'w') as f:
    json.dump(concepts_data, f, indent=2)

print(f'Saved {len(concepts_data)} concepts to output/concepts.json')

---
**Next:** [Notebook 3 — Relation Extraction](03_relation_extraction.ipynb)